# Week 1 checks: data loader + perturbation preview

Runs the whole Week 1 pipeline and writes figures to `figures/week1/<tag>/`.

* If `PETCT_DATA_ROOT` (or `data_root` in `configs/week1.yaml`) is set, it runs on **your real data**.
* Otherwise it runs on a **synthetic phantom** (tag `phantom`) so you can check that everything works.
  Phantom figures only show the code works; they say nothing about real images.

The perturbations are synthetic image-quality changes and are **not** claimed to represent any particular scanner.

In [ ]:
import os, sys, tempfile, json
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir() and (p / "configs").is_dir()), Path.cwd())
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

from src import data, perturb, viz
from src.phantom import make_phantom_dataset

cfg = data.load_config("configs/week1.yaml")
print("project folder:", ROOT)

## 1. Data source

In [ ]:
data_root = data.resolve_data_root(cfg)
if data_root is None:
    TAG = "phantom"
    data_root = make_phantom_dataset(tempfile.mkdtemp(), n_patients=12, seed=0)
    cfg["patch"]["size"] = [32, 32, 32]          # phantom volumes are 48x48x40
    print("No data root set -> using a SYNTHETIC PHANTOM (pipeline check only).")
else:
    TAG = "real"
    print("Using real data at:", data_root)
FIG = ROOT / "figures" / "week1" / TAG
FIG.mkdir(parents=True, exist_ok=True)

## 2. Index and patient-disjoint split

In [ ]:
RES = ROOT / "results" / "week1"
if TAG == "real" and (RES / "index.csv").exists():      # cached index (lesion voxel counts already computed)
    df = data.load_index_cache(data_root, RES / "index.csv")
else:
    df = data.index_dataset(data_root, cfg)
    if TAG == "real":
        RES.mkdir(parents=True, exist_ok=True)
        df.assign(**{k: [os.path.relpath(v, data_root).replace(os.sep, "/") for v in df[k]] for k in ("pet", "ct", "mask")}).to_csv(RES / "index.csv", index=False)
df = data.apply_max_cases(df, cfg.get("max_cases"), cfg["seed"])
split = data.make_split(df, cfg["split"], cfg["seed"])
data.assert_no_leakage(split)                    # raises if any patient is in two splits

# real data -> splits.json (re-used by Weeks 2-4); phantom -> separate file so it never overwrites it
split_file = RES / "splits.json" if TAG == "real" else Path(tempfile.mkdtemp()) / "splits_phantom.json"
if TAG == "real" and split_file.exists():            # the split is frozen: never regenerate it
    assert data.load_split(split_file) == split, "splits.json differs from a fresh split; do not overwrite it"
else:
    data.save_split(split, df, cfg, split_file)

rows = []
for name, pids in split.items():
    sub = df[df["patient_id"].isin(pids)]
    rows.append({"split": name, "patients": len(pids), "studies": len(sub),
                 "lesion-positive studies": int((sub["lesion_voxels"] > 0).sum()),
                 "lesion-free studies": int((sub["lesion_voxels"] == 0).sum())})
display(pd.DataFrame(rows))
print("split file:", split_file, "| no patient appears in more than one split")

## 3. Loader: patch mode (training) and full-volume mode (evaluation)

In [ ]:
train_ds = data.build_dataset(data.records_for(df, split, "train"), data.train_transforms(cfg), cfg["cache_rate"])
batch = next(iter(data.build_loader(train_ds, batch_size=1, shuffle=True)))
print("patch batch  image:", tuple(batch["image"].shape), " mask:", tuple(batch["mask"].shape), " (channels = [PET, CT])")

test_recs = data.records_for(df, split, "test")
full = data.load_clean_case(test_recs[0], cfg)
print("full volume  image:", tuple(full["image"].shape), " mask:", tuple(full["mask"].shape))

## 4. Perturbation conditions (frozen)

Levels come from `configs/week1.yaml`. Run `python -m src.freeze` once before any evaluation; after that
this cell fails if the levels ever change.

In [ ]:
specs = perturb.specs_from_config(cfg)
display(pd.DataFrame([s.to_dict() for s in specs]))
frozen = json.loads((ROOT / "configs" / "perturbations_frozen.json").read_text())
assert frozen["fingerprint"] == perturb.fingerprint(specs), "levels differ from the frozen record!"
print("levels match configs/perturbations_frozen.json:", frozen["fingerprint"][:12], "...")

## 5. Figures for representative lesion-positive test cases

In [ ]:
fc = ROOT / "results" / "week1" / "figure_cases.csv"
if TAG == "real" and fc.exists():                       # the 3 cases shown in figures/week1/real (lymphoma, lung, melanoma)
    ids = set(pd.read_csv(fc)["case_id"]); pos = [r for r in test_recs if r["case_id"] in ids]
else:
    pos = [r for r in test_recs if r["lesion_voxels"] > 0][:3]
assert pos, "no lesion-positive test cases (increase max_cases or change the seed)"
made = []
for i, rec in enumerate(pos):
    s = data.load_clean_case(rec, cfg)
    tag = f"case{i + 1}"
    made += [viz.perturbation_grid(s, specs, 0, FIG / f"{tag}_grid_pet.png"),
             viz.perturbation_grid(s, specs, 1, FIG / f"{tag}_grid_ct.png"),
             viz.mip_figure(s, specs, FIG / f"{tag}_mip.png"),
             viz.difference_figure(s, specs, FIG / f"{tag}_diff.png"),
             viz.histogram_figure(s, specs, FIG / f"{tag}_hist.png")]
print(len(made), "figures written to", FIG)
for p in made[:5]:
    display(Image(str(p)))

## 6. Sanity checks on these cases

In [ ]:
import torch
def check_case(rec):
    s = data.load_clean_case(rec, cfg)
    img, mask = s["image"].as_tensor(), s["mask"].as_tensor()
    assert img.shape[0] == 2 and img.shape[1:] == mask.shape[1:], "image/mask grid differs"
    assert set(torch.unique(mask).tolist()) <= {0.0, 1.0}
    for sp in specs:
        out = perturb.apply_to_sample(sp, s, rec["case_id"])
        assert out["image"].shape == img.shape, sp.name
        assert out["mask"] is s["mask"], sp.name                      # mask never touched
        assert int(out["mask"].sum()) == int(mask.sum()), sp.name
    assert torch.equal(perturb.apply_spec(specs[0], img, "x"), img)   # clean == clean
    pr = perturb.remove_pet(img)
    assert torch.equal(pr[1], img[1]) and float(pr[0].abs().max()) == 0.0
    return True

print({rec["case_id"]: check_case(rec) for rec in pos})

In [ ]:
import subprocess
r = subprocess.run([sys.executable, "-m", "pytest", "tests", "-q", "--disable-warnings"], capture_output=True, text=True, cwd=ROOT)
print(r.stdout[-600:]); print(r.stderr[-300:])